# Advanced Analysis — Questions 11-15

**Techniques:** `RFM Segmentation`, `Cohort Analysis`, `Statistical Testing`, `Composite Scoring`, `MoM Growth with LAG()`

| # | Question |
|---|----------|
| Q11 | RFM Analysis — Customer Segmentation |
| Q12 | Cohort Analysis — Monthly Retention |
| Q13 | Late delivery impact on review scores (+ statistical test) |
| Q14 | Seller composite performance score |
| Q15 | Month-over-Month revenue growth by category |

In [ ]:
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from src.db_utils import run_query

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## Q11: RFM Customer Segmentation

In [ ]:
q11 = run_query("""
    WITH rfm_base AS (
        SELECT c.customer_unique_id,
               MAX(o.order_purchase_timestamp) AS last_purchase,
               COUNT(DISTINCT o.order_id)      AS frequency,
               ROUND(SUM(op.payment_value), 2) AS monetary
        FROM customers c
        JOIN orders o          ON c.customer_id = o.customer_id
        JOIN order_payments op ON o.order_id = op.order_id
        WHERE o.order_status = 'delivered'
        GROUP BY c.customer_unique_id
    ),
    rfm_scores AS (
        SELECT *,
               NTILE(5) OVER (ORDER BY last_purchase DESC) AS r_score,
               NTILE(5) OVER (ORDER BY frequency ASC)     AS f_score,
               NTILE(5) OVER (ORDER BY monetary ASC)      AS m_score
        FROM rfm_base
    )
    SELECT *,
           (r_score + f_score + m_score) AS rfm_total,
           CASE
               WHEN (r_score + f_score + m_score) >= 13 THEN 'Champions'
               WHEN (r_score + f_score + m_score) >= 10 THEN 'Loyal Customers'
               WHEN (r_score + f_score + m_score) >= 7  THEN 'Potential Loyalists'
               WHEN (r_score + f_score + m_score) >= 4  THEN 'At Risk'
               ELSE 'Lost'
           END AS segment
    FROM rfm_scores
    ORDER BY rfm_total DESC
""")

segment_summary = q11.groupby('segment').agg(
    customer_count=('customer_unique_id', 'count'),
    avg_monetary=('monetary', 'mean'),
    avg_frequency=('frequency', 'mean')
).round(2).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = {'Champions': '#2ecc71', 'Loyal Customers': '#3498db',
          'Potential Loyalists': '#f39c12', 'At Risk': '#e67e22', 'Lost': '#e74c3c'}
segment_order = ['Champions', 'Loyal Customers', 'Potential Loyalists', 'At Risk', 'Lost']

segment_summary_sorted = segment_summary.set_index('segment').reindex(segment_order).reset_index()
bar_colors = [colors[s] for s in segment_summary_sorted['segment']]

axes[0].bar(segment_summary_sorted['segment'], segment_summary_sorted['customer_count'], color=bar_colors)
axes[0].set_title('Customer Count by RFM Segment')
axes[0].set_xticklabels(segment_summary_sorted['segment'], rotation=20, ha='right')
axes[0].set_ylabel('Customer Count')

axes[1].bar(segment_summary_sorted['segment'], segment_summary_sorted['avg_monetary'], color=bar_colors)
axes[1].set_title('Avg Spending by RFM Segment (BRL)')
axes[1].set_xticklabels(segment_summary_sorted['segment'], rotation=20, ha='right')
axes[1].set_ylabel('Avg Monetary Value')

plt.suptitle('RFM Customer Segmentation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/q11_rfm_segmentation.png', bbox_inches='tight')
plt.show()
segment_summary_sorted

## Q12: Cohort Analysis — Monthly Retention

In [ ]:
q12 = run_query("""
    WITH cohort_base AS (
        SELECT c.customer_unique_id,
               STRFTIME('%Y-%m', MIN(o.order_purchase_timestamp)) AS cohort_month,
               STRFTIME('%Y-%m', o.order_purchase_timestamp)      AS order_month
        FROM customers c
        JOIN orders o ON c.customer_id = o.customer_id
        WHERE o.order_status = 'delivered'
        GROUP BY c.customer_unique_id, order_month
    ),
    cohort_size AS (
        SELECT cohort_month, COUNT(DISTINCT customer_unique_id) AS cohort_customers
        FROM cohort_base GROUP BY cohort_month
    ),
    retention AS (
        SELECT cb.cohort_month, cb.order_month,
               COUNT(DISTINCT cb.customer_unique_id) AS active_customers
        FROM cohort_base cb
        GROUP BY cb.cohort_month, cb.order_month
    )
    SELECT r.cohort_month, r.order_month, cs.cohort_customers, r.active_customers,
           ROUND(r.active_customers * 100.0 / cs.cohort_customers, 2) AS retention_rate
    FROM retention r
    JOIN cohort_size cs ON r.cohort_month = cs.cohort_month
    ORDER BY r.cohort_month, r.order_month
""")

pivot = q12.pivot_table(index='cohort_month', columns='order_month',
                        values='retention_rate', aggfunc='first')

plt.figure(figsize=(16, 8))
sns.heatmap(pivot.iloc[:12, :12], annot=True, fmt='.1f', cmap='YlOrRd_r',
            linewidths=0.5, cbar_kws={'label': 'Retention Rate (%)'})
plt.title('Cohort Retention Heatmap (%)', fontsize=14, fontweight='bold')
plt.xlabel('Order Month')
plt.ylabel('Cohort Month')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../data/q12_cohort_retention.png', bbox_inches='tight')
plt.show()
print(f'Cohorts analyzed: {q12["cohort_month"].nunique()}')

## Q13: Late Delivery Impact on Review Scores
### SQL analysis + Python statistical significance test (Welch's t-test)

In [ ]:
q13 = run_query("""
    WITH delivery_status AS (
        SELECT
            CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
                 THEN 'Late' ELSE 'On Time' END AS delivery_flag,
            r.review_score
        FROM orders o
        JOIN order_reviews r ON o.order_id = r.order_id
        WHERE o.order_status = 'delivered'
          AND o.order_delivered_customer_date IS NOT NULL
    )
    SELECT delivery_flag, COUNT(*) AS total_orders,
           ROUND(AVG(review_score), 3) AS avg_score,
           SUM(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END) AS low_scores,
           ROUND(SUM(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS low_score_pct
    FROM delivery_status
    GROUP BY delivery_flag
""")

# Detailed scores for t-test
scores_raw = run_query("""
    SELECT
        CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
             THEN 'Late' ELSE 'On Time' END AS delivery_flag,
        CAST(r.review_score AS FLOAT) AS review_score
    FROM orders o
    JOIN order_reviews r ON o.order_id = r.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
""")

on_time_scores = scores_raw[scores_raw['delivery_flag'] == 'On Time']['review_score']
late_scores    = scores_raw[scores_raw['delivery_flag'] == 'Late']['review_score']

t_stat, p_value = stats.ttest_ind(on_time_scores, late_scores, equal_var=False)

print('=== Welch\'s t-test: On Time vs Late Delivery ===')
print(f't-statistic : {t_stat:.4f}')
print(f'p-value     : {p_value:.2e}')
print(f'Result      : {"STATISTICALLY SIGNIFICANT" if p_value < 0.05 else "NOT significant"} (α=0.05)')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.barplot(data=q13, x='delivery_flag', y='avg_score', ax=axes[0],
            palette={'On Time': '#2ecc71', 'Late': '#e74c3c'})
axes[0].set_title('Avg Review Score: On Time vs Late')
axes[0].set_ylim(0, 5)
axes[0].set_ylabel('Average Score')
axes[0].set_xlabel('')

sns.barplot(data=q13, x='delivery_flag', y='low_score_pct', ax=axes[1],
            palette={'On Time': '#2ecc71', 'Late': '#e74c3c'})
axes[1].set_title('% Orders with Score ≤ 2')
axes[1].set_ylabel('Low Score %')
axes[1].set_xlabel('')

plt.suptitle('Late Delivery Impact on Customer Satisfaction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/q13_delivery_vs_score.png', bbox_inches='tight')
plt.show()
q13

## Q14: Seller Composite Performance Score

In [ ]:
q14 = run_query("""
    WITH seller_stats AS (
        SELECT oi.seller_id,
               COUNT(DISTINCT oi.order_id)  AS total_orders,
               ROUND(SUM(oi.price), 2)      AS total_revenue,
               ROUND(AVG(r.review_score), 3) AS avg_review,
               ROUND(AVG(JULIANDAY(o.order_delivered_customer_date) -
                         JULIANDAY(o.order_purchase_timestamp)), 1) AS avg_delivery_days
        FROM order_items oi
        JOIN orders o        ON oi.order_id = o.order_id
        JOIN order_reviews r ON o.order_id = r.order_id
        WHERE o.order_status = 'delivered'
          AND o.order_delivered_customer_date IS NOT NULL
        GROUP BY oi.seller_id
        HAVING total_orders >= 10
    ),
    scored AS (
        SELECT *,
               NTILE(5) OVER (ORDER BY total_revenue ASC)      AS volume_score,
               NTILE(5) OVER (ORDER BY avg_review ASC)         AS review_score,
               NTILE(5) OVER (ORDER BY avg_delivery_days DESC) AS speed_score
        FROM seller_stats
    )
    SELECT seller_id, total_orders, total_revenue, avg_review, avg_delivery_days,
           ROUND(volume_score * 0.4 + review_score * 0.4 + speed_score * 0.2, 2) AS composite_score
    FROM scored
    ORDER BY composite_score DESC
    LIMIT 20
""")

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(range(len(q14)), q14['composite_score'],
               color=plt.cm.RdYlGn(q14['composite_score'] / 5))
ax.set_yticks(range(len(q14)))
ax.set_yticklabels([s[:12] + '...' for s in q14['seller_id']], fontsize=8)
ax.set_xlabel('Composite Score (max 5.0)')
ax.set_title('Top 20 Sellers by Composite Performance Score\n(40% Volume + 40% Review + 20% Speed)',
             fontsize=12, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../data/q14_seller_scores.png', bbox_inches='tight')
plt.show()
q14.head(10)

## Q15: Month-over-Month Revenue Growth by Category

In [ ]:
q15 = run_query("""
    WITH monthly_cat_revenue AS (
        SELECT STRFTIME('%Y-%m', o.order_purchase_timestamp) AS month,
               t.product_category_name_english AS category,
               ROUND(SUM(oi.price), 2)         AS revenue
        FROM orders o
        JOIN order_items oi ON o.order_id = oi.order_id
        JOIN products p     ON oi.product_id = p.product_id
        JOIN product_category_name_translation t
            ON p.product_category_name = t.product_category_name
        WHERE o.order_status = 'delivered'
        GROUP BY month, category
    ),
    with_lag AS (
        SELECT month, category, revenue,
               LAG(revenue) OVER (PARTITION BY category ORDER BY month) AS prev_month_revenue
        FROM monthly_cat_revenue
    )
    SELECT month, category, revenue, prev_month_revenue,
           ROUND((revenue - prev_month_revenue) * 100.0 / NULLIF(prev_month_revenue, 0), 2) AS mom_growth_pct
    FROM with_lag
    WHERE prev_month_revenue IS NOT NULL
    ORDER BY mom_growth_pct DESC
    LIMIT 30
""")

top_growth = q15.head(10)
fig, ax = plt.subplots(figsize=(11, 6))
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in top_growth['mom_growth_pct']]
ax.barh([f"{r['category']} ({r['month']})" for _, r in top_growth.iterrows()],
        top_growth['mom_growth_pct'], color=colors)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlabel('MoM Growth (%)')
ax.set_title('Top 10 Category-Months by MoM Revenue Growth', fontsize=13, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../data/q15_mom_growth.png', bbox_inches='tight')
plt.show()
q15.head(10)